# CountVectorizer 모델과 코사인 유사도를 이용한 영화 추천 서비스

## 데이터 불러오기

In [ ]:
# colab과 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd

# 파일 경로 설정
file_path='/content/drive/MyDrive/sk/movies_metadata.csv'

# DataFrame 생성
df = pd.read_csv(file_path)

# 결과 확인
display(df)

/tmp/ipykernel_3684/17280763.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45461,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,NaN,0.0,90.0,"[{'iso_639_1': 'fa', 'name': 'فارسی'}]",Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0
45462,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,2011-11-17,0.0,360.0,"[{'iso_639_1': 'tl', 'name': ''}]",Released,NaN,Century of Birthing,False,9.0,3.0
45463,False,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",NaN,67758,tt0303758,en,Betrayal,"When one of her hits goes wrong, a professiona...",...,2003-08-01,0.0,90.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,A deadly game of wits.,Betrayal,False,3.8,6.0
45464,False,NaN,0,[],NaN,227506,tt0008536,en,Satana likuyushchiy,"In a small town live two brothers, one a minis...",...,1917-10-21,0.0,87.0,[],Released,NaN,Satan Triumphant,False,0.0,0.0


## 전처리

In [ ]:
# overview, title 컬럼 -> 팬시 인덱싱
df1 = df[['overview', 'title']]
display(df1)

,overview,title
0,"Led by Woody, Andy's toys live happily in his ...",Toy Story
1,When siblings Judy and Peter discover an encha...,Jumanji
2,A family wedding reignites the ancient feud be...,Grumpier Old Men
3,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale
4,Just when George Banks has recovered from his ...,Father of the Bride Part II
...,...,...
45461,Rising and falling between a man and woman.,Subdue
45462,An artist struggles to finish his work while a...,Century of Birthing
45463,"When one of her hits goes wrong, a professiona...",Betrayal
45464,"In a small town live two brothers, one a minis...",Satan Triumphant


In [ ]:
# 누락 데이터 처리

## 각 컬럼별 누락 데이터의 수 확인
num_nulls = df1.isnull().sum()
print(f'각 컬럼별 누락 데이터의 수: \n{num_nulls}')

print('-'*80)

## 누락 데이터가 존재하는 행 제거
df2 = df1.dropna(ignore_index=True)
display(df2)

각 컬럼별 누락 데이터의 수: 
overview    954
title         6
dtype: int64
--------------------------------------------------------------------------------


,overview,title
0,"Led by Woody, Andy's toys live happily in his ...",Toy Story
1,When siblings Judy and Peter discover an encha...,Jumanji
2,A family wedding reignites the ancient feud be...,Grumpier Old Men
3,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale
4,Just when George Banks has recovered from his ...,Father of the Bride Part II
...,...,...
44501,Rising and falling between a man and woman.,Subdue
44502,An artist struggles to finish his work while a...,Century of Birthing
44503,"When one of her hits goes wrong, a professiona...",Betrayal
44504,"In a small town live two brothers, one a minis...",Satan Triumphant


## 텍스트 임베딩

In [ ]:
# 필요한 클래스 임포트
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# CountVectorizer 호출, 임베딩 모델 생성
cv = CountVectorizer(stop_words='english')

In [ ]:
# 영어 텍스트 데이터 생성 / 단어 사전 및 임베딩 행렬 생성

## data의 수 -> 10000개 축소(행: 0 ~ 9999)
df3 = df2[0:10000]
display(df3)

print('-'*80)

## 단어 사전 생성에 사용할 텍스트 데이터(말뭉치) 생성
text_list = df3['overview'].tolist()
print(text_list)

print('-'*80)

## fit() 함수 -> 단어 사전 생성
cv.fit(text_list)

## 단어 사전 확인
movies_vocab = cv.vocabulary_
print(f'생성된 영어 단어 사전: \n{movies_vocab}')
print('-'*80)
print(f'생성된 영어 단어의 수: {len(movies_vocab)}개')

print('-'*80)

# 임베딩 행렬 생성
embedding_matrix = cv.transform(text_list).toarray()
print(f'overview 컬럼에 대한 임베딩의 결과(모양): {embedding_matrix.shape}')

,overview,title
0,"Led by Woody, Andy's toys live happily in his ...",Toy Story
1,When siblings Judy and Peter discover an encha...,Jumanji
2,A family wedding reignites the ancient feud be...,Grumpier Old Men
3,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale
4,Just when George Banks has recovered from his ...,Father of the Bride Part II
...,...,...
9995,"In the peaceful countryside, Vassily opposes t...",Earth
9996,"Francisco is rich, rather strict on principles...",Él
9997,"Cashier Maurice Legrand is married to Adele, a...",La Chienne
9998,"While Erendira, a beautiful teenage girl, has ...",Eréndira


--------------------------------------------------------------------------------
["Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.", "When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.", "A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local bait shop, alarming the locals who w

In [ ]:
# 10000개 영화 줄거리(overview) -> 코사인 유사도 측정

## 필요한 함수 임포트
from sklearn.metrics.pairwise import cosine_similarity

## cosine_similarity() 함수 실행
sim = cosine_similarity(embedding_matrix, embedding_matrix)

# 결과 확인(10000행, 10000열)
print(sim)

[[1.         0.02153652 0.         ... 0.         0.         0.        ]
 [0.02153652 1.         0.05170877 ... 0.         0.         0.        ]
 [0.         0.05170877 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]


## 영화 추천 서비스 구현

In [ ]:
# 코사인 유사도 결과 값 -> DataFrame 생성

## 컬럼과 행 인덱스 설정
index=df3['title'].values
columns=df3['title'].values

## DataFrame 생성
df_sim = pd.DataFrame(data=sim, index=index, columns=columns)

## 결과 확인
display(df_sim)

,Toy Story,Jumanji,Grumpier Old Men,Waiting to Exhale,Father of the Bride Part II,Heat,Sabrina,Tom and Huck,Sudden Death,GoldenEye,...,Between Your Legs,Zatôichi Meets the One-Armed Swordsman,The Angry Silence,Isadora,San Francisco,Earth,Él,La Chienne,Eréndira,The Private Lives of Elizabeth and Essex
Toy Story,1.000000,0.021537,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.020838,0.0,0.000000,0.039621,0.000000,0.000000,0.000000,0.000000,0.000000
Jumanji,0.021537,1.000000,0.051709,0.000000,0.000000,0.081230,0.000000,0.000000,0.164536,0.0,...,0.017645,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Grumpier Old Men,0.000000,0.051709,1.000000,0.000000,0.031846,0.000000,0.000000,0.024015,0.000000,0.0,...,0.000000,0.000000,0.0,0.030802,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Waiting to Exhale,0.000000,0.000000,0.000000,1.000000,0.000000,0.035921,0.000000,0.028006,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.063246
Father of the Bride Part II,0.000000,0.000000,0.031846,0.000000,1.000000,0.000000,0.047946,0.000000,0.067557,0.0,...,0.021734,0.000000,0.0,0.000000,0.000000,0.000000,0.030949,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Earth,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,1.000000,0.055556,0.000000,0.047619,0.000000
Él,0.000000,0.000000,0.000000,0.000000,0.030949,0.000000,0.000000,0.000000,0.020211,0.0,...,0.058521,0.000000,0.0,0.000000,0.000000,0.055556,1.000000,0.025126,0.023810,0.000000
La Chienne,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.017645,0.000000,0.0,0.054153,0.000000,0.000000,0.025126,1.000000,0.000000,0.000000
Eréndira,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.016720,0.000000,0.0,0.000000,0.039621,0.047619,0.023810,0.000000,1.000000,0.000000


In [ ]:
# 특정 영화 지정 -> 특정 컬럼 지정(예: Toy Story)

## 컬럼의 값 -> 크기 순으로 정렬(내림차순) -> 해당 영화는 제외, 나머지에서 top_n 추출
n = 10
top_n = df_sim['Toy Story'].sort_values(ascending=False)[1:n+1]
print(f'Toy Story와 줄거리가 가장 유사한 영화 top10: \n{top_n}')

Toy Story와 줄거리가 가장 유사한 영화 top10: 
Toy Story 2                                    0.412393
The Champ                                      0.245402
Rivers and Tides                               0.142857
Man on the Moon                                0.142857
Class of 1984                                  0.139047
Malice                                         0.131095
Heartbeeps                                     0.129219
Condorman                                      0.127775
Rebel Without a Cause                          0.127000
The First $20 Million Is Always the Hardest    0.123718
Name: Toy Story, dtype: float64


In [ ]:
# 특정 영화 기준 -> 줄거리가 유사한 영화 추천 함수 생성
def top_n_recommend(k, title):
    ## 사용자의 입력이 10000개의 영화 제목에 있는 경우에만 실행
    if title in df_sim.columns:
        top_n = df_sim[title].sort_values(ascending=False)[1:k+1]
        return top_n
    else:
        print("존재하지 않는 영화 제목이오니, 확인 후 다시 입력을 해주세요")

In [ ]:
# 영화 추천 함수 실행

## 매개 변수 값 설정
num = 10
name = 'GoldenEye'

## 함수 호출
top_10 = top_n_recommend(k=num, title=name)

## 결과 확인
print(top_10)

Licence to Kill          0.240563
Never Say Never Again    0.196960
You Only Live Twice      0.195217
Casino Royale            0.195217
Octopussy                0.185695
Live and Let Die         0.181902
Phantasm                 0.181902
Dr. No                   0.174078
Diamonds Are Forever     0.156386
Thunderball              0.153093
Name: GoldenEye, dtype: float64
